# Il ruolo del successo femminile nel medagliere olimpico

Questo notebook aggiunge una prospettiva complementare al progetto **"Non solo Giochi"**: invece di chiedersi soltanto quali fattori economici e demografici spiegano il numero totale di medaglie, osserva quanto il successo delle atlete donne contribuisce al medagliere dei paesi.

L'analisi usa i dati granulari atleta-evento e le biografie degli atleti per rispondere a tre domande:

1. la quota di atlete donne e la quota di medaglie femminili sono cresciute nel tempo?
2. quali paesi hanno costruito una parte rilevante del proprio successo olimpico sulle gare femminili?
3. esistono paesi in cui il rendimento femminile e' superiore alla quota di partecipazione femminile?

L'obiettivo e' descrittivo: non dimostra causalita', ma misura una dimensione qualitativa del successo olimpico che non emerge dai soli indicatori macroeconomici.

In [1]:
from pathlib import Path
import re

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.3f}".format)

# I grafici usano dataset aggregati, ma disattivo comunque il limite righe di Altair
# per evitare errori quando si sperimentano filtri diversi.
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## 1. Caricamento dei dati

Uso due dataset:

- `Olympic_Athlete_Event_Details.csv`: una riga per atleta-evento, con sport, evento, NOC, medaglia e indicatore di sport di squadra;
- `Olympic_Athlete_Biography.csv`: biografia degli atleti, usata soprattutto per ricavare il sesso dell'atleta e il nome del paese associato al NOC.

Il notebook cerca automaticamente la root del progetto risalendo dalle cartelle correnti, cosi' puo' essere eseguito sia dalla root del repository sia dalla cartella del notebook.

In [2]:
def find_project_root() -> Path:
    """Trova la cartella del progetto a partire dalla working directory del notebook."""
    expected = Path("data") / "csv_olimpiadi" / "Olympic_Athlete_Event_Details.csv"
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / expected).exists():
            return base
    raise FileNotFoundError(
        "Non trovo la cartella data/csv_olimpiadi. "
        "Esegui il notebook dalla root del progetto o da una sua sottocartella."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "csv_olimpiadi"

EVENTS_PATH = DATA_DIR / "Olympic_Athlete_Event_Details.csv"
BIO_PATH = DATA_DIR / "Olympic_Athlete_Biography.csv"

print("Project root:", PROJECT_ROOT)
print("Event details:", EVENTS_PATH.name)
print("Biographies:", BIO_PATH.name)

Project root: C:\Users\smacchiavelli\development\code\master\progettone
Event details: Olympic_Athlete_Event_Details.csv
Biographies: Olympic_Athlete_Biography.csv


In [3]:
event_cols = [
    "edition", "country_noc", "sport", "event",
    "athlete_id", "medal", "isTeamSport"
]
bio_cols = ["athlete_id", "name", "sex", "country", "country_noc"]

events = pd.read_csv(EVENTS_PATH, usecols=event_cols)
bio = pd.read_csv(BIO_PATH, usecols=bio_cols)

# Porto gli ID allo stesso tipo prima del merge.
events["athlete_id"] = events["athlete_id"].astype("string")
bio["athlete_id"] = bio["athlete_id"].astype("string")

print(f"Righe atleta-evento: {len(events):,}")
print(f"Atleti in biografia: {bio['athlete_id'].nunique():,}")
display(events.head())

Righe atleta-evento: 316,834
Atleti in biografia: 155,861


,edition,country_noc,sport,event,athlete_id,medal,isTeamSport
0,1908 Summer Olympics,ANZ,Athletics,"100 metres, Men",64710,NaN,False
1,1908 Summer Olympics,ANZ,Athletics,"400 metres, Men",64756,NaN,False
2,1908 Summer Olympics,ANZ,Athletics,"800 metres, Men",64808,NaN,False
3,1908 Summer Olympics,ANZ,Athletics,"800 metres, Men",922519,NaN,False
4,1908 Summer Olympics,ANZ,Athletics,"800 metres, Men",64735,NaN,False


## 2. Preparazione: solo Olimpiadi estive dal 1964 al 2020

Il resto del progetto lavora soprattutto sulle Olimpiadi estive dal 1964 in avanti. Qui mantengo lo stesso perimetro e arrivo fino a Tokyo 2020, che e' l'ultima edizione estiva presente nei file atleta-evento.

Per le medaglie bisogna fare attenzione agli sport di squadra: nel file atleta-evento una medaglia di squadra compare una volta per ogni atleta della squadra. Per il medagliere, invece, va contata una sola volta per paese, evento e tipo di medaglia.

In [4]:
# Estraggo l'anno dall'etichetta dell'edizione, es. "1964 Summer Olympics".
events["year"] = events["edition"].str.extract(r"(\d{4})", expand=False).astype(int)
events["is_summer"] = events["edition"].str.contains("Summer Olympics", case=False, na=False)

events = events.loc[events["is_summer"] & events["year"].between(1964, 2020)].copy()

# Normalizzo il flag degli sport di squadra: puo' arrivare come bool o come stringa.
if events["isTeamSport"].dtype == "bool":
    events["is_team_sport"] = events["isTeamSport"]
else:
    events["is_team_sport"] = (
        events["isTeamSport"].astype("string").str.strip().str.lower().eq("true")
    )

events = events.merge(
    bio[["athlete_id", "sex"]],
    on="athlete_id",
    how="left",
)

events["sex"] = events["sex"].fillna("Unknown")
events["has_medal"] = events["medal"].notna() & events["medal"].astype("string").str.strip().ne("")

print(f"Righe dopo filtro Summer 1964-2020: {len(events):,}")
print("Anni:", int(events["year"].min()), "-", int(events["year"].max()))
print("Distribuzione sesso atleta:")
display(events["sex"].value_counts(dropna=False).to_frame("righe_atleta_evento"))

Righe dopo filtro Summer 1964-2020: 179,141
Anni: 1964 - 2020
Distribuzione sesso atleta:


,righe_atleta_evento
sex,
Male,116863
Female,62278


In [5]:
def classify_event_gender(event: str) -> str:
    """Classifica il genere dell'evento dal nome dell'evento olimpico."""
    text = str(event).lower()
    if re.search(r"\bwomen\b|women's|, women", text):
        return "Female"
    if re.search(r"\bmen\b|men's|, men", text):
        return "Male"
    return "Mixed/Other"

# Per il medagliere uso il genere dell'evento, non il sesso della singola riga atleta.
# Cosi' una medaglia di squadra femminile viene contata una sola volta come medaglia femminile.
events["event_gender"] = events["event"].apply(classify_event_gender)

print("Eventi per genere dell'evento:")
display(
    events[["sport", "event", "event_gender"]]
    .drop_duplicates()
    ["event_gender"].value_counts()
    .to_frame("eventi_distinti")
)

Eventi per genere dell'evento:


,eventi_distinti
event_gender,
Male,232
Female,186
Mixed/Other,39


## 3. Partecipazione femminile e medaglie femminili

Costruisco due indicatori distinti:

- **quota di atlete donne**: percentuale di atlete uniche donne tra tutti gli atleti di un paese in una edizione;
- **quota di medaglie femminili**: percentuale di medaglie vinte in eventi femminili sul totale delle medaglie vinte in eventi maschili + femminili.

Gli eventi `Mixed/Other` vengono tenuti nel conteggio totale ma non entrano nella quota femminile, per non attribuire arbitrariamente una medaglia mista a un solo genere.

In [6]:
# Partecipazione: conto atleti unici per paese, anno e sesso.
athletes_unique = (
    events.dropna(subset=["athlete_id"])
    .drop_duplicates(["year", "country_noc", "athlete_id", "sex"])
)

participation = (
    athletes_unique
    .groupby(["year", "country_noc", "sex"], as_index=False)
    .agg(athletes=("athlete_id", "nunique"))
    .pivot_table(
        index=["year", "country_noc"],
        columns="sex",
        values="athletes",
        fill_value=0,
        aggfunc="sum",
    )
    .reset_index()
)

for col in ["Female", "Male", "Unknown"]:
    if col not in participation.columns:
        participation[col] = 0

participation = participation.rename(
    columns={
        "Female": "female_athletes",
        "Male": "male_athletes",
        "Unknown": "unknown_athletes",
    }
)
participation["total_athletes"] = (
    participation["female_athletes"]
    + participation["male_athletes"]
    + participation["unknown_athletes"]
)
participation["female_athlete_share"] = np.where(
    participation["total_athletes"] > 0,
    participation["female_athletes"] / participation["total_athletes"],
    np.nan,
)

display(participation.head())

sex,year,country_noc,female_athletes,male_athletes,unknown_athletes,total_athletes,female_athlete_share
0,1964,AFG,0,8,0,8,0.000
1,1964,AHO,0,4,0,4,0.000
2,1964,ALG,0,1,0,1,0.000
3,1964,ARG,8,101,0,109,0.073
4,1964,AUS,41,205,0,246,0.167


In [7]:
# Medagliere: conto una sola medaglia per squadra negli sport di squadra.
medal_rows = events.loc[events["has_medal"]].copy()

team_medals = (
    medal_rows.loc[medal_rows["is_team_sport"]]
    .drop_duplicates(["year", "country_noc", "sport", "event", "medal"])
)
individual_medals = medal_rows.loc[~medal_rows["is_team_sport"]].copy()

medal_units = pd.concat([individual_medals, team_medals], ignore_index=True)

medals_by_gender = (
    medal_units
    .groupby(["year", "country_noc", "event_gender"], as_index=False)
    .size()
    .rename(columns={"size": "medals"})
    .pivot_table(
        index=["year", "country_noc"],
        columns="event_gender",
        values="medals",
        fill_value=0,
        aggfunc="sum",
    )
    .reset_index()
)

for col in ["Female", "Male", "Mixed/Other"]:
    if col not in medals_by_gender.columns:
        medals_by_gender[col] = 0

medals_by_gender = medals_by_gender.rename(
    columns={
        "Female": "female_medals",
        "Male": "male_medals",
        "Mixed/Other": "mixed_other_medals",
    }
)
medals_by_gender["total_medals"] = (
    medals_by_gender["female_medals"]
    + medals_by_gender["male_medals"]
    + medals_by_gender["mixed_other_medals"]
)
medals_by_gender["female_medal_share"] = np.where(
    (medals_by_gender["female_medals"] + medals_by_gender["male_medals"]) > 0,
    medals_by_gender["female_medals"]
    / (medals_by_gender["female_medals"] + medals_by_gender["male_medals"]),
    np.nan,
)

display(medals_by_gender.head())
print(f"Medaglie/evento contate dopo deduplica squadre: {medal_units.shape[0]:,}")

event_gender,year,country_noc,female_medals,male_medals,mixed_other_medals,total_medals,female_medal_share
0,1964,ARG,0,0,1,1,NaN
1,1964,AUS,7,10,1,18,0.412
2,1964,BAH,0,0,1,1,NaN
3,1964,BEL,0,3,0,3,0.000
4,1964,BRA,0,1,0,1,0.000


Medaglie/evento contate dopo deduplica squadre: 11,781


In [8]:
# Ricavo un nome paese leggibile dal file biografico.
country_lookup = (
    bio.dropna(subset=["country_noc", "country"])
    .assign(country=lambda d: d["country"].astype("string").str.strip())
    .groupby("country_noc")["country"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0])
    .reset_index()
)

panel = participation.merge(
    medals_by_gender,
    on=["year", "country_noc"],
    how="outer",
).merge(country_lookup, on="country_noc", how="left")

panel["country"] = panel["country"].fillna(panel["country_noc"])

count_cols = [
    "female_athletes", "male_athletes", "unknown_athletes", "total_athletes",
    "female_medals", "male_medals", "mixed_other_medals", "total_medals",
]
for col in count_cols:
    panel[col] = panel[col].fillna(0)

panel["female_athlete_share"] = np.where(
    panel["total_athletes"] > 0,
    panel["female_athletes"] / panel["total_athletes"],
    np.nan,
)
panel["female_medal_share"] = np.where(
    (panel["female_medals"] + panel["male_medals"]) > 0,
    panel["female_medals"] / (panel["female_medals"] + panel["male_medals"]),
    np.nan,
)

# Un indice semplice: valori >1 indicano che la quota di medaglie femminili
# e' superiore alla quota di atlete donne nella delegazione.
panel["female_success_index"] = panel["female_medal_share"] / panel["female_athlete_share"]
panel = panel.sort_values(["year", "country"])

display(panel.head())
print(f"Osservazioni paese-anno: {len(panel):,}")

,year,country_noc,female_athletes,male_athletes,unknown_athletes,total_athletes,female_athlete_share,female_medals,male_medals,mixed_other_medals,total_medals,female_medal_share,country,female_success_index
0,1964,AFG,0,8,0,8,0.000,0.000,0.000,0.000,0.000,NaN,Afghanistan,NaN
2,1964,ALG,0,1,0,1,0.000,0.000,0.000,0.000,0.000,NaN,Algeria,NaN
3,1964,ARG,8,101,0,109,0.073,0.000,0.000,1.000,1.000,NaN,Argentina,NaN
4,1964,AUS,41,205,0,246,0.167,7.000,10.000,1.000,18.000,0.412,Australia,2.471
5,1964,AUT,11,46,0,57,0.193,0.000,0.000,0.000,0.000,NaN,Austria,NaN


Osservazioni paese-anno: 2,386


## 4. Andamento globale: partecipazione e medaglie

Il primo grafico confronta due curve globali:

- la quota di atlete donne tra tutti gli atleti;
- la quota di medaglie assegnate a paesi in eventi femminili.

Se la seconda curva cresce insieme alla prima, significa che l'ampliamento della partecipazione femminile non e' solo numerico: entra anche nel medagliere.

In [9]:
global_year = (
    panel.groupby("year", as_index=False)
    .agg(
        female_athletes=("female_athletes", "sum"),
        male_athletes=("male_athletes", "sum"),
        unknown_athletes=("unknown_athletes", "sum"),
        female_medals=("female_medals", "sum"),
        male_medals=("male_medals", "sum"),
        mixed_other_medals=("mixed_other_medals", "sum"),
        total_medals=("total_medals", "sum"),
    )
)

global_year["female_athlete_share"] = (
    global_year["female_athletes"]
    / (global_year["female_athletes"] + global_year["male_athletes"] + global_year["unknown_athletes"])
)
global_year["female_medal_share"] = (
    global_year["female_medals"]
    / (global_year["female_medals"] + global_year["male_medals"])
)

trend_long = global_year.melt(
    id_vars="year",
    value_vars=["female_athlete_share", "female_medal_share"],
    var_name="indicatore",
    value_name="quota",
)
trend_long["indicatore"] = trend_long["indicatore"].map({
    "female_athlete_share": "Quota atlete donne",
    "female_medal_share": "Quota medaglie femminili",
})

trend_chart = (
    alt.Chart(trend_long)
    .mark_line(point=True, strokeWidth=2.5)
    .encode(
        x=alt.X("year:O", title="Edizione olimpica"),
        y=alt.Y("quota:Q", title="Quota", axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0, 0.6])),
        color=alt.Color("indicatore:N", title="Indicatore"),
        tooltip=[
            alt.Tooltip("year:O", title="Anno"),
            alt.Tooltip("indicatore:N", title="Indicatore"),
            alt.Tooltip("quota:Q", title="Quota", format=".1%"),
        ],
    )
    .properties(
        width=760,
        height=380,
        title="Crescita globale della partecipazione femminile e delle medaglie femminili",
    )
)

trend_chart

alt.Chart(...)

## 5. Quali paesi vincono piu' medaglie femminili nel periodo recente?

Qui restringo l'attenzione dal 2000 in poi: e' il periodo piu' vicino al contesto contemporaneo e riduce il peso delle edizioni storiche in cui il programma femminile era piu' limitato.

In [10]:
recent = panel.loc[panel["year"] >= 2000].copy()

country_recent = (
    recent.groupby(["country_noc", "country"], as_index=False)
    .agg(
        female_medals=("female_medals", "sum"),
        male_medals=("male_medals", "sum"),
        mixed_other_medals=("mixed_other_medals", "sum"),
        total_medals=("total_medals", "sum"),
        female_athletes=("female_athletes", "sum"),
        male_athletes=("male_athletes", "sum"),
        total_athletes=("total_athletes", "sum"),
    )
)
country_recent["female_medal_share"] = np.where(
    (country_recent["female_medals"] + country_recent["male_medals"]) > 0,
    country_recent["female_medals"]
    / (country_recent["female_medals"] + country_recent["male_medals"]),
    np.nan,
)
country_recent["female_athlete_share"] = np.where(
    country_recent["total_athletes"] > 0,
    country_recent["female_athletes"] / country_recent["total_athletes"],
    np.nan,
)
country_recent["female_success_index"] = (
    country_recent["female_medal_share"] / country_recent["female_athlete_share"]
)

top_female = country_recent.nlargest(15, "female_medals").sort_values("female_medals")

bar_chart = (
    alt.Chart(top_female)
    .mark_bar()
    .encode(
        x=alt.X("female_medals:Q", title="Medaglie in eventi femminili"),
        y=alt.Y("country:N", sort="x", title="Paese"),
        color=alt.Color("female_medal_share:Q", title="Quota femminile", scale=alt.Scale(scheme="reds")),
        tooltip=[
            alt.Tooltip("country:N", title="Paese"),
            alt.Tooltip("female_medals:Q", title="Medaglie femminili", format=",.0f"),
            alt.Tooltip("total_medals:Q", title="Medaglie totali", format=",.0f"),
            alt.Tooltip("female_medal_share:Q", title="Quota femminile", format=".1%"),
        ],
    )
    .properties(
        width=760,
        height=420,
        title="Top 15 paesi per medaglie in eventi femminili, 2000-2020",
    )
)

bar_chart

alt.Chart(...)

## 6. Partecipazione femminile vs rendimento femminile

Questo scatter confronta, per ogni paese dal 2000 in poi:

- asse X: quota di atlete donne nella delegazione;
- asse Y: quota di medaglie vinte in eventi femminili;
- dimensione: totale medaglie vinte.

I paesi sopra la diagonale hanno una quota di medaglie femminili superiore alla quota di atlete donne: in termini descrittivi, il contributo femminile al medagliere e' particolarmente forte.

In [11]:
scatter_data = country_recent.loc[
    (country_recent["total_medals"] >= 5)
    & country_recent["female_athlete_share"].notna()
    & country_recent["female_medal_share"].notna()
].copy()

base = alt.Chart(scatter_data)

points = base.mark_circle(opacity=0.72, stroke="#333", strokeWidth=0.4).encode(
    x=alt.X(
        "female_athlete_share:Q",
        title="Quota atlete donne nella delegazione",
        axis=alt.Axis(format="%"),
        scale=alt.Scale(domain=[0, 0.7]),
    ),
    y=alt.Y(
        "female_medal_share:Q",
        title="Quota medaglie in eventi femminili",
        axis=alt.Axis(format="%"),
        scale=alt.Scale(domain=[0, 0.9]),
    ),
    size=alt.Size(
        "total_medals:Q",
        title="Medaglie totali",
        scale=alt.Scale(range=[40, 900]),
    ),
    color=alt.Color(
        "female_success_index:Q",
        title="Indice successo femminile",
        scale=alt.Scale(scheme="tealblues"),
    ),
    tooltip=[
        alt.Tooltip("country:N", title="Paese"),
        alt.Tooltip("female_athlete_share:Q", title="Quota atlete", format=".1%"),
        alt.Tooltip("female_medal_share:Q", title="Quota medaglie femm.", format=".1%"),
        alt.Tooltip("female_success_index:Q", title="Indice successo", format=".2f"),
        alt.Tooltip("female_medals:Q", title="Medaglie femm.", format=",.0f"),
        alt.Tooltip("total_medals:Q", title="Medaglie totali", format=",.0f"),
    ],
)

diagonal = alt.Chart(pd.DataFrame({"x": [0, 0.7], "y": [0, 0.7]})).mark_line(
    strokeDash=[5, 5], color="#777"
).encode(x="x:Q", y="y:Q")

scatter_chart = (diagonal + points).properties(
    width=760,
    height=440,
    title="Partecipazione femminile e contributo femminile al medagliere, 2000-2020",
).interactive()

scatter_chart

alt.LayerChart(...)

## 7. Paesi dove il contributo femminile pesa di piu'

La tabella seguente non premia soltanto chi vince tante medaglie. Considera paesi con almeno 10 medaglie dal 2000 in poi e ordina per **indice di successo femminile**:

\[
\text{indice} = \frac{\text{quota medaglie femminili}}{\text{quota atlete donne}}
\]

Un valore maggiore di 1 indica che le atlete donne contribuiscono al medagliere piu' di quanto pesino numericamente nella delegazione.

In [12]:
specialists = (
    country_recent.loc[
        (country_recent["total_medals"] >= 10)
        & country_recent["female_success_index"].replace([np.inf, -np.inf], np.nan).notna()
    ]
    .sort_values("female_success_index", ascending=False)
    .head(15)
    .copy()
)

cols = [
    "country", "female_medals", "male_medals", "total_medals",
    "female_athlete_share", "female_medal_share", "female_success_index",
]

display(
    specialists[cols]
    .assign(
        female_athlete_share=lambda d: d["female_athlete_share"].map(lambda x: f"{x:.1%}"),
        female_medal_share=lambda d: d["female_medal_share"].map(lambda x: f"{x:.1%}"),
        female_success_index=lambda d: d["female_success_index"].map(lambda x: f"{x:.2f}"),
    )
    .rename(columns={
        "country": "Paese",
        "female_medals": "Medaglie femm.",
        "male_medals": "Medaglie masch.",
        "total_medals": "Medaglie totali",
        "female_athlete_share": "Quota atlete",
        "female_medal_share": "Quota medaglie femm.",
        "female_success_index": "Indice successo femm.",
    })
)

,Paese,Medaglie femm.,Medaglie masch.,Medaglie totali,Quota atlete,Quota medaglie femm.,Indice successo femm.
18,Belgium,13.000,12.000,26.000,36.4%,52.0%,1.43
124,Mexico,17.000,13.000,31.000,39.8%,56.7%,1.42
138,Netherlands,80.000,45.000,138.000,46.2%,64.0%,1.39
188,Thailand,19.000,10.000,29.000,48.4%,65.5%,1.35
116,Lithuania,10.000,13.000,23.000,32.6%,43.5%,1.33
154,Poland,36.000,34.000,71.000,38.7%,51.4%,1.33
57,Egypt,7.000,13.000,20.000,26.7%,35.0%,1.31
180,Switzerland,22.000,20.000,45.000,40.1%,52.4%,1.31
160,Romania,47.000,22.000,69.000,53.1%,68.1%,1.28
7,Argentina,10.000,14.000,27.000,32.9%,41.7%,1.27


## 8. Evoluzione per le principali nazioni

La heatmap mostra come cambia la quota di medaglie femminili nel tempo per i paesi con piu' medaglie femminili dal 2000 in poi. Serve a distinguere due casi diversi:

- paesi storicamente forti anche nelle competizioni femminili;
- paesi in cui il contributo femminile cresce solo nelle edizioni piu' recenti.

In [13]:
top_nocs = country_recent.nlargest(12, "female_medals")["country_noc"].tolist()
heat_data = panel.loc[panel["country_noc"].isin(top_nocs)].copy()
heat_data = heat_data.loc[heat_data["total_medals"] > 0].copy()
heat_data["female_medal_share_label"] = heat_data["female_medal_share"].fillna(0)

heatmap = (
    alt.Chart(heat_data)
    .mark_rect()
    .encode(
        x=alt.X("year:O", title="Anno"),
        y=alt.Y("country:N", title="Paese", sort="-x"),
        color=alt.Color(
            "female_medal_share_label:Q",
            title="Quota medaglie femm.",
            scale=alt.Scale(scheme="redpurple", domain=[0, 0.8]),
        ),
        tooltip=[
            alt.Tooltip("country:N", title="Paese"),
            alt.Tooltip("year:O", title="Anno"),
            alt.Tooltip("female_medals:Q", title="Medaglie femm.", format=",.0f"),
            alt.Tooltip("total_medals:Q", title="Medaglie totali", format=",.0f"),
            alt.Tooltip("female_medal_share_label:Q", title="Quota femm.", format=".1%"),
        ],
    )
    .properties(
        width=760,
        height=330,
        title="Quota di medaglie femminili nelle principali nazioni, 1964-2020",
    )
)

heatmap

alt.Chart(...)

## 9. Sintesi automatica dei risultati

Questa cella produce alcune frasi riassuntive basate sui dati calcolati sopra. E' utile per trasformare l'analisi in testo da relazione o presentazione.

In [14]:
first_year = int(global_year["year"].min())
last_year = int(global_year["year"].max())
first = global_year.loc[global_year["year"] == first_year].iloc[0]
last = global_year.loc[global_year["year"] == last_year].iloc[0]

top_country = country_recent.sort_values("female_medals", ascending=False).iloc[0]
top_share = (
    country_recent.loc[country_recent["total_medals"] >= 10]
    .sort_values("female_medal_share", ascending=False)
    .iloc[0]
)
top_index = specialists.iloc[0]

print("Sintesi")
print("-------")
print(
    f"Dal {first_year} al {last_year}, la quota globale di atlete donne passa "
    f"da {first['female_athlete_share']:.1%} a {last['female_athlete_share']:.1%}."
)
print(
    f"Nello stesso periodo, la quota di medaglie in eventi femminili passa "
    f"da {first['female_medal_share']:.1%} a {last['female_medal_share']:.1%}."
)
print(
    f"Dal 2000 in poi, il paese con piu' medaglie femminili e' "
    f"{top_country['country']} ({top_country['female_medals']:.0f} medaglie)."
)
print(
    f"Tra i paesi con almeno 10 medaglie, la quota femminile piu' alta e' "
    f"{top_share['country']} ({top_share['female_medal_share']:.1%})."
)
print(
    f"L'indice di successo femminile piu' alto nella tabella e' "
    f"{top_index['country']} ({top_index['female_success_index']:.2f})."
)

Sintesi
-------
Dal 1964 al 2020, la quota globale di atlete donne passa da 13.0% a 47.8%.
Nello stesso periodo, la quota di medaglie in eventi femminili passa da 21.0% a 48.3%.
Dal 2000 in poi, il paese con piu' medaglie femminili e' United States (317 medaglie).
Tra i paesi con almeno 10 medaglie, la quota femminile piu' alta e' Romania (68.1%).
L'indice di successo femminile piu' alto nella tabella e' Belgium (1.43).


## Conclusione

Questa analisi mostra che il successo olimpico non dipende solo dalla dimensione economica o demografica del paese. La capacita' di includere, formare e valorizzare le atlete donne e' una componente concreta del medagliere.

Nel progetto complessivo, questa sezione puo' essere usata come esempio di **fattore qualitativo**: due paesi con risorse simili possono ottenere risultati diversi anche per la diversa profondita' del loro sistema sportivo femminile, per le discipline su cui investono e per il grado di accesso delle donne allo sport competitivo.

Limiti principali:

- la classificazione delle medaglie usa il genere dell'evento ricavato dal nome dell'evento;
- gli eventi misti sono tenuti separati per evitare attribuzioni arbitrarie;
- l'indice di successo femminile e' descrittivo e non va letto come misura causale.